In [1]:
!pip install --upgrade nbformat
!pip install nbformat ipywidgets

In [8]:
import pandas as pd
import requests
from bs4 import BeautifulSoup
import plotly.express as px
import re
from datetime import datetime, timedelta

def magavail(start_date, end_date, mags=["pg0", "pg1"]):
    base_url = "https://themis.ssl.berkeley.edu/data/themis/thg/l2/mag/"
    all_data = []

    # Get range of years to search
    years = range(start_date.year, end_date.year + 1)

    for mag in mags:
        for year in years:
            url = f"{base_url}{mag}/{year}/"
            try:
                response = requests.get(url, timeout=10)
                if response.status_code != 200:
                    continue

                soup = BeautifulSoup(response.text, 'html.parser')
                # Pattern: thg_l2_mag_pg0_20160101_v01.cdf
                pattern = re.compile(rf"thg_l2_mag_{mag}_(\d{{8}})_v\d{{2}}\.cdf")

                for link in soup.find_all('a'):
                    match = pattern.search(link.get('href', ''))
                    if match:
                        date_str = match.group(1)
                        file_date = datetime.strptime(date_str, "%Y%m%d")

                        if start_date <= file_date <= end_date:
                            all_data.append({'Mag': mag, 'Date': file_date})
            except Exception as e:
                print(f"Error accessing {url}: {e}")

    if not all_data:
        return pd.DataFrame()

    df = pd.DataFrame(all_data)
    df = df.sort_values(['Mag', 'Date'])

    # Consolidate consecutive dates into ranges for better Gantt visualization
    df['grp'] = (df['Date'] - df.groupby('Mag')['Date'].shift(1) > timedelta(days=1)).cumsum()

    gantt_df = df.groupby(['Mag', 'grp']).agg(
        Start=('Date', 'min'),
        Finish=('Date', 'max')
    ).reset_index()

    # Add 1 day to finish to make the bar span the full day
    gantt_df['Finish'] = gantt_df['Finish'] + timedelta(days=1)

    return gantt_df

# --- Usage ---
start = datetime(2004, 1, 1)
end = datetime(2026, 12, 31)

df = magavail(start, end, mags=["pg0", "pg1", "pg2", "pg3", "pg4", "pg5"])

if not df.empty:
    fig = px.timeline(
        df,
        x_start="Start",
        x_end="Finish",
        y="Mag",
        color="Mag",
        title="AALPIP Data Availability"
    )
    # fig.update_yaxes(autorange="reversed") # Better for Gantt/Timeline
    fig.write_html("output/AALPIP Availability.html")
    fig.show()
else:
    print("No data found for the given criteria.")

In [3]:
df

,Mag,grp,Start,Finish
0,pg0,0,2016-01-01,2016-02-08
1,pg0,1,2016-02-09,2016-06-11
2,pg0,2,2016-10-14,2017-01-01
3,pg1,2,2016-01-01,2016-07-21
4,pg1,3,2016-10-22,2017-01-01
